# 03 - SQL Analysis with DuckDB

## Purpose

This notebook uses DuckDB to run SQL analysis on the cleaned sales dataset.

The goal is to calculate business KPIs and prepare query outputs that can support an executive sales dashboard.

In [1]:
import duckdb
from pathlib import Path

processed_path = Path("../data/processed")
clean_sales_file = processed_path / "clean_sales.csv"

In [2]:
con = duckdb.connect()

In [3]:
con.execute(f"""
CREATE OR REPLACE VIEW clean_sales AS
SELECT *
FROM read_csv(
    '{clean_sales_file}',
    header = true,
    columns = {{
        'invoiceno': 'VARCHAR',
        'stockcode': 'VARCHAR',
        'description': 'VARCHAR',
        'quantity': 'INTEGER',
        'invoicedate': 'TIMESTAMP',
        'unitprice': 'DOUBLE',
        'customerid': 'VARCHAR',
        'country': 'VARCHAR',
        'revenue': 'DOUBLE',
        'transaction_type': 'VARCHAR'
    }}
)
""")

In [4]:
con.execute("""
SELECT *
FROM clean_sales
LIMIT 5
""").df()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,Sale
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Sale
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,Sale
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Sale
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Sale


In [5]:
con.execute("""
DESCRIBE clean_sales""").df()

,column_name,column_type,null,key,default,extra
0,invoiceno,VARCHAR,YES,None,None,None
1,stockcode,VARCHAR,YES,None,None,None
2,description,VARCHAR,YES,None,None,None
3,quantity,INTEGER,YES,None,None,None
4,invoicedate,TIMESTAMP,YES,None,None,None
5,unitprice,DOUBLE,YES,None,None,None
6,customerid,VARCHAR,YES,None,None,None
7,country,VARCHAR,YES,None,None,None
8,revenue,DOUBLE,YES,None,None,None
9,transaction_type,VARCHAR,YES,None,None,None


In [6]:
con.execute("""
SELECT
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoiceno) AS total_orders,
    COUNT(*) AS total_line_items,
    SUM(quantity) AS total_quantity_sold,
    ROUND(SUM(revenue) / COUNT(DISTINCT invoiceno), 2) AS average_order_value
FROM clean_sales
""").df()

,total_revenue,total_orders,total_line_items,total_quantity_sold,average_order_value
0,10631048.74,19959,524877,5572419.0,532.64


In [7]:
con.execute("""
SELECT
    DATE_TRUNC('month', invoicedate::TIMESTAMP) AS sales_month,
    ROUND(SUM(revenue), 2) AS monthly_revenue,
    COUNT(DISTINCT invoiceno) AS monthly_orders,
    SUM(quantity) AS monthly_quantity_sold
FROM clean_sales
GROUP BY sales_month
ORDER BY sales_month
""").df()

,sales_month,monthly_revenue,monthly_orders,monthly_quantity_sold
0,2010-12-01,821452.73,1559,358019.0
1,2011-01-01,689811.61,1086,387099.0
2,2011-02-01,522545.56,1100,282934.0
3,2011-03-01,716215.26,1454,376599.0
4,2011-04-01,536968.49,1246,307953.0
5,2011-05-01,769296.61,1681,395001.0
6,2011-06-01,760547.01,1533,388511.0
7,2011-07-01,718076.12,1475,399693.0
8,2011-08-01,746779.32,1360,421019.0
9,2011-09-01,1056435.19,1837,569573.0


In [8]:
con.execute("""
SELECT
    stockcode,
    description,
    ROUND(SUM(revenue), 2) AS total_revenue,
    SUM(quantity) AS total_quantity_sold,
    COUNT(DISTINCT invoiceno) AS order_count
FROM clean_sales
GROUP BY stockcode, description
ORDER BY total_revenue DESC
LIMIT 10
""").df()

,stockcode,description,total_revenue,total_quantity_sold,order_count
0,DOT,DOTCOM POSTAGE,206248.77,706.0,706
1,22423,REGENCY CAKESTAND 3 TIER,174156.54,13851.0,1988
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995.0,1
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104284.24,37580.0,2189
4,47566,PARTY BUNTING,99445.23,18283.0,1685
5,85099B,JUMBO BAG RED RETROSPOT,94159.81,48371.0,2089
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033.0,247
7,POST,POSTAGE,78101.88,3150.0,1126
8,M,Manual,77750.27,6984.0,289
9,23084,RABBIT NIGHT LIGHT,66870.03,30739.0,994


## Export Dashboard-Ready Summary Tables

This section exports SQL query results as summary CSV files that can be used directly for dashboard creation.

Instead of connecting dashboard tools to raw line-level data only, we prepare business-friendly summary tables for KPIs, trends, products, countries, and customers.

In [9]:
dashboard_data_path = Path("../data/processed/dashboard")
dashboard_data_path.mkdir(parents=True, exist_ok=True)

In [10]:
executive_kpis = con.execute("""
SELECT
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoiceno) AS total_orders,
    COUNT(*) AS total_line_items,
    SUM(quantity) AS total_quantity_sold,
    ROUND(SUM(revenue) / COUNT(DISTINCT invoiceno), 2) AS average_order_value
FROM clean_sales
""").df()

executive_kpis.to_csv(dashboard_data_path / "executive_kpis.csv", index=False)

executive_kpis

,total_revenue,total_orders,total_line_items,total_quantity_sold,average_order_value
0,10631048.74,19959,524877,5572419.0,532.64


In [11]:
monthly_revenue = con.execute("""
SELECT
    DATE_TRUNC('month', invoicedate::TIMESTAMP) AS sales_month,
    ROUND(SUM(revenue), 2) AS monthly_revenue,
    COUNT(DISTINCT invoiceno) AS monthly_orders,
    SUM(quantity) AS monthly_quantity_sold
FROM clean_sales
GROUP BY sales_month
ORDER BY sales_month
""").df()

monthly_revenue.to_csv(dashboard_data_path / "monthly_revenue.csv", index=False)

monthly_revenue

,sales_month,monthly_revenue,monthly_orders,monthly_quantity_sold
0,2010-12-01,821452.73,1559,358019.0
1,2011-01-01,689811.61,1086,387099.0
2,2011-02-01,522545.56,1100,282934.0
3,2011-03-01,716215.26,1454,376599.0
4,2011-04-01,536968.49,1246,307953.0
5,2011-05-01,769296.61,1681,395001.0
6,2011-06-01,760547.01,1533,388511.0
7,2011-07-01,718076.12,1475,399693.0
8,2011-08-01,746779.32,1360,421019.0
9,2011-09-01,1056435.19,1837,569573.0


In [12]:
top_products = con.execute("""
SELECT
    stockcode,
    description,
    ROUND(SUM(revenue), 2) AS total_revenue,
    SUM(quantity) AS total_quantity_sold,
    COUNT(DISTINCT invoiceno) AS order_count
FROM clean_sales
GROUP BY stockcode, description
ORDER BY total_revenue DESC
LIMIT 10
""").df()

top_products.to_csv(dashboard_data_path / "top_products.csv", index=False)

top_products

,stockcode,description,total_revenue,total_quantity_sold,order_count
0,DOT,DOTCOM POSTAGE,206248.77,706.0,706
1,22423,REGENCY CAKESTAND 3 TIER,174156.54,13851.0,1988
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995.0,1
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104284.24,37580.0,2189
4,47566,PARTY BUNTING,99445.23,18283.0,1685
5,85099B,JUMBO BAG RED RETROSPOT,94159.81,48371.0,2089
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033.0,247
7,POST,POSTAGE,78101.88,3150.0,1126
8,M,Manual,77750.27,6984.0,289
9,23084,RABBIT NIGHT LIGHT,66870.03,30739.0,994


In [13]:
country_revenue = con.execute("""
SELECT
    country,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoiceno) AS total_orders,
    SUM(quantity) AS total_quantity_sold,
    ROUND(SUM(revenue) / COUNT(DISTINCT invoiceno), 2) AS average_order_value
FROM clean_sales
GROUP BY country
ORDER BY total_revenue DESC
""").df()

country_revenue.to_csv(dashboard_data_path / "country_revenue.csv", index=False)

country_revenue.head(10)

,country,total_revenue,total_orders,total_quantity_sold,average_order_value
0,United Kingdom,8990682.03,18018,4646905.0,498.98
1,Netherlands,285446.34,94,200361.0,3036.66
2,EIRE,283140.52,288,147007.0,983.13
3,Germany,228678.40,457,119154.0,500.39
4,France,209625.37,392,112060.0,534.76
5,Australia,138453.81,57,83891.0,2429.01
6,Spain,61558.56,90,27933.0,683.98
7,Switzerland,57067.60,54,30617.0,1056.81
8,Belgium,41196.34,98,23237.0,420.37
9,Sweden,38367.83,36,36078.0,1065.77


In [14]:
customer_revenue = con.execute("""
SELECT
    customerid,
    country,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoiceno) AS total_orders,
    SUM(quantity) AS total_quantity_sold,
    ROUND(SUM(revenue) / COUNT(DISTINCT invoiceno), 2) AS average_order_value
FROM clean_sales
WHERE customerid IS NOT NULL
GROUP BY customerid, country
ORDER BY total_revenue DESC
LIMIT 20
""").df()

customer_revenue.to_csv(dashboard_data_path / "customer_revenue.csv", index=False)

customer_revenue

,customerid,country,total_revenue,total_orders,total_quantity_sold,average_order_value
0,14646,Netherlands,280206.02,73,196915.0,3838.44
1,18102,United Kingdom,259657.30,60,64124.0,4327.62
2,17450,United Kingdom,194390.79,46,69973.0,4225.89
3,16446,United Kingdom,168472.50,2,80997.0,84236.25
4,14911,EIRE,143711.17,201,80240.0,714.98
5,12415,Australia,124914.53,21,77374.0,5948.31
6,14156,EIRE,117210.08,55,57768.0,2131.09
7,17511,United Kingdom,91062.38,31,64549.0,2937.50
8,16029,United Kingdom,80850.84,63,40108.0,1283.35
9,12346,United Kingdom,77183.60,1,74215.0,77183.60
